# Vidal et al. (2019) Case 2: Full-to-Full Transfer

Case 2 transfers only the source Case 1 `model_state_dict` into a fresh target Vidal LSTM. The target optimizer, scheduler, and min-max scaler are recreated; all paper trainable parameters remain unlocked during target fine-tuning.

All run cells are gated. By default this notebook performs setup and preflight only; it does not automatically start smoke, diagnostic, or formal training.

## 1. Runtime and repository setup

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_ROOT = Path("/content/battery-soc-transfer-benchmark")
DRIVE_ROOT = Path("/content/drive/MyDrive/battery_soc_benchmark")
VIDAL_V2_ROOT = DRIVE_ROOT / 'data/labelled/vidal_sequence_v2'
CASE1_OUTPUT_ROOT = DRIVE_ROOT / 'outputs/vidal_case1'
CASE2_OUTPUT_ROOT = DRIVE_ROOT / 'outputs/vidal_case2'
LOCAL_CACHE_ROOT = Path('/content/battery_soc_cache/vidal_case2')
REPO_URL = "https://github.com/zhangcastle/battery-soc-transfer-benchmark.git"

def is_valid_repo(path: Path) -> bool:
    return (
        (path / '.git').is_dir()
        and (path / 'pyproject.toml').is_file()
    )

if REPO_ROOT.exists() and not is_valid_repo(REPO_ROOT):
    shutil.rmtree(REPO_ROOT)
    print('Removed incomplete repository directory')

if not is_valid_repo(REPO_ROOT):
    from google.colab import userdata

    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError(
            'Missing Colab secret GITHUB_TOKEN. '
            'Add a fine-grained GitHub token with read access to '
            'zhangcastle/battery-soc-transfer-benchmark and enable Notebook access.'
        )
    authenticated_url = (
        f"https://x-access-token:{github_token}"
        "@github.com/zhangcastle/battery-soc-transfer-benchmark.git"
    )
    try:
        subprocess.run(
            [
                'git',
                'clone',
                '--depth',
                '1',
                authenticated_url,
                str(REPO_ROOT),
            ],
            check=True,
            text=True,
        )
        subprocess.run(
            [
                'git',
                'remote',
                'set-url',
                'origin',
                REPO_URL,
            ],
            cwd=REPO_ROOT,
            check=True,
            text=True,
        )
    except subprocess.CalledProcessError:
        if REPO_ROOT.exists() and not is_valid_repo(REPO_ROOT):
            shutil.rmtree(REPO_ROOT)
        raise RuntimeError(
            'Authenticated GitHub clone failed. Check that GITHUB_TOKEN '
            'has read access to zhangcastle/battery-soc-transfer-benchmark '
            'and that Notebook access is enabled.'
        ) from None
    finally:
        del authenticated_url
        del github_token
else:
    existing_commit = subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True
    ).strip()
    print('Existing repository detected; not pulling or overwriting local changes.')
    print('existing commit:', existing_commit)

subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-e',
        str(REPO_ROOT),
    ],
    check=True,
    text=True,
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

current_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
print('Paths configured:')
print('repo root:', REPO_ROOT)
print('current working directory:', Path.cwd())
print('current commit:', current_commit)
print('Drive input root:', VIDAL_V2_ROOT)
print('Case 1 output root:', CASE1_OUTPUT_ROOT)
print('Case 2 output root:', CASE2_OUTPUT_ROOT)


## 2. Git and environment preflight

Formal runs require a known commit and a clean working tree. If `dirty=True`, restart the runtime or restore tracked files before formal execution.

In [ ]:
import torch
import yaml
import pandas as pd
from IPython.display import Image, display

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
dirty_text = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_ROOT, text=True).strip()
dirty = bool(dirty_text)
print('commit:', commit)
print('dirty:', dirty)
if dirty:
    print(dirty_text)
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
print('device_name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. SOC v2 input preflight

In [ ]:
from battery_benchmark.data.vidal_sequence_v2_dataset import validate_case1_roots

checks = validate_case1_roots(VIDAL_V2_ROOT)
display(checks)
assert (checks['status'] == 'PASS').all(), 'Required SOC v2 inputs are missing.'

## 4. Case 1 formal checkpoint discovery

Only formal Case 1 runs with `epoch_515.pt`, actual checkpoint epoch 514, and `metrics.json` are valid Case 2 sources.

In [ ]:
def discover_case1_formal_runs() -> pd.DataFrame:
    rows = []
    for dataset in ['panasonic_18650pf', 'lg_hg2', 'samsung_30t']:
        root = CASE1_OUTPUT_ROOT / dataset / 'seed_42'
        for meta_path in sorted(root.glob('*/run_metadata.json')):
            run = meta_path.parent
            metadata = json.loads(meta_path.read_text())
            ckpt = run / 'checkpoints' / 'epoch_515.pt'
            metrics = run / 'metrics.json'
            epoch = None
            if ckpt.is_file():
                try:
                    epoch = int(torch.load(ckpt, map_location='cpu', weights_only=False).get('epoch'))
                except Exception as exc:
                    epoch = 'ERROR: ' + type(exc).__name__
            rows.append({
                'dataset': dataset,
                'run_path': str(run),
                'checkpoint_path': str(ckpt),
                'checkpoint_epoch': epoch,
                'run_mode': metadata.get('run_mode'),
                'development_only': metadata.get('development_only'),
                'checkpoint_exists': ckpt.is_file(),
                'metrics_exists': metrics.is_file(),
                'git_commit': metadata.get('git_commit'),
            })
    return pd.DataFrame(rows)

case1_runs = discover_case1_formal_runs()
if case1_runs.empty:
    print('No Case 1 runs found under:', CASE1_OUTPUT_ROOT)
else:
    display(case1_runs)

formal = case1_runs[
    (case1_runs['run_mode'] == 'formal')
    & (case1_runs['development_only'] == False)
    & (case1_runs['checkpoint_exists'])
    & (case1_runs['metrics_exists'])
    & (case1_runs['checkpoint_epoch'] == 514)
].copy() if not case1_runs.empty else pd.DataFrame()

display(formal)
print('valid formal source checkpoints:', len(formal))

## 5. Quick smoke gated execution

This is an integration smoke only: 2 epochs, at most 32 target windows, batch size 32, output under `_smoke`. It is not a performance result.

In [ ]:

SOURCE = 'lg_hg2'
TARGET = 'samsung_30t'
RUN_QUICK_SMOKE = False

match = formal[formal['dataset'] == SOURCE]
assert not match.empty, 'Missing formal Case 1 source checkpoint for ' + SOURCE
SOURCE_CHECKPOINT = Path(match.iloc[-1]['checkpoint_path'])
print('source checkpoint:', SOURCE_CHECKPOINT)

if RUN_QUICK_SMOKE:
    subprocess.run([
        sys.executable, 'scripts/run_vidal_case2.py',
        '--source', SOURCE,
        '--target', TARGET,
        '--config', 'configs/experiments/vidal_case2_smoke.yaml',
        '--input-root', str(VIDAL_V2_ROOT),
        '--output-root', str(CASE2_OUTPUT_ROOT),
        '--case1-output-root', str(CASE1_OUTPUT_ROOT),
        '--source-checkpoint', str(SOURCE_CHECKPOINT),
        '--cache-root', str(LOCAL_CACHE_ROOT),
        '--run-mode', 'smoke',
        '--device', 'auto',
        '--max-train-windows', '32',
    ], cwd=REPO_ROOT, check=True, text=True)
else:
    print('Set RUN_QUICK_SMOKE = True to run the 2-epoch quick smoke.')


## 6. Inspect latest quick smoke

In [ ]:
def load_resolved_config(run: Path) -> dict:
    path = run / 'resolved_config.yaml'
    if not path.is_file():
        return {}
    return yaml.safe_load(path.read_text()) or {}

def matching_runs(pair_root: Path, predicate):
    matches = []
    for run in sorted(pair_root.glob('*'), key=lambda p: p.stat().st_mtime):
        cfg = load_resolved_config(run)
        if cfg and predicate(cfg):
            matches.append(run)
    return matches

pair_root = CASE2_OUTPUT_ROOT / '_smoke' / 'source_lg_hg2__target_samsung_30t' / 'seed_42'
quick_runs = matching_runs(
    pair_root,
    lambda cfg: cfg.get('run_mode') == 'smoke'
    and cfg.get('epochs') == 2
    and cfg.get('batch_size') == 32
    and cfg.get('max_train_windows') == 32
    and cfg.get('development_only') is True,
)
if not quick_runs:
    print('No matching 2-epoch quick smoke run found.')
else:
    latest_quick = quick_runs[-1]
    print('latest quick smoke:', latest_quick)
    metadata = json.loads((latest_quick / 'run_metadata.json').read_text())
    metrics = json.loads((latest_quick / 'metrics.json').read_text())
    print('source checkpoint:', metadata['source_checkpoint']['source_checkpoint_path'])
    print('source checkpoint epoch:', metadata['source_checkpoint']['source_checkpoint_epoch'])
    print('actual_train_window_count:', metadata['actual_train_window_count'])
    print('initial_mse_window_count:', metadata['initial_mse_window_count'])
    print('initial target MSE:', metadata['initial_target_mse_before_training'])
    print('final target train MSE:', metadata['final_target_train_mse'])
    print('optimizer_reinitialized:', metadata['optimizer_reinitialized'])
    print('scheduler_reinitialized:', metadata['scheduler_reinitialized'])
    print('target_scaler_refit:', metadata['target_scaler_refit'])
    print('device:', metadata['device'])
    print('metrics:', metrics)
    for figure in sorted((latest_quick / 'figures').glob('*.png')):
        display(Image(filename=str(figure)))

## 7. Diagnostic gated execution

This development-only diagnostic uses all target training windows for 5 epochs with batch size 512. It is useful for checking training stability, still writes under `_smoke`, and is not a formal run.

In [ ]:

RUN_DIAGNOSTIC = False

if RUN_DIAGNOSTIC:
    subprocess.run([
        sys.executable, 'scripts/run_vidal_case2.py',
        '--source', SOURCE,
        '--target', TARGET,
        '--config', 'configs/experiments/vidal_case2_diagnostic.yaml',
        '--input-root', str(VIDAL_V2_ROOT),
        '--output-root', str(CASE2_OUTPUT_ROOT),
        '--case1-output-root', str(CASE1_OUTPUT_ROOT),
        '--source-checkpoint', str(SOURCE_CHECKPOINT),
        '--cache-root', str(LOCAL_CACHE_ROOT),
        '--run-mode', 'smoke',
        '--device', 'auto',
    ], cwd=REPO_ROOT, check=True, text=True)
else:
    print('Set RUN_DIAGNOSTIC = True to run the 5-epoch full-window diagnostic.')


## 8. Inspect latest diagnostic

In [ ]:
diagnostic_runs = matching_runs(
    pair_root,
    lambda cfg: cfg.get('run_mode') == 'smoke'
    and cfg.get('epochs') == 5
    and cfg.get('batch_size') == 512
    and cfg.get('max_train_windows') is None
    and cfg.get('development_only') is True,
)
if not diagnostic_runs:
    print('No 5-epoch full-window diagnostic run found yet.')
else:
    latest_diagnostic = diagnostic_runs[-1]
    print('latest diagnostic:', latest_diagnostic)
    metadata = json.loads((latest_diagnostic / 'run_metadata.json').read_text())
    metrics = json.loads((latest_diagnostic / 'metrics.json').read_text())
    print('run_dir:', latest_diagnostic)
    print('git_commit:', metadata['git_commit'])
    print('git_dirty:', metadata['git_dirty'])
    print('device:', metadata['device'])
    print('actual_train_window_count:', metadata['actual_train_window_count'])
    print('initial_mse_window_count:', metadata['initial_mse_window_count'])
    print('initial target MSE:', metadata['initial_target_mse_before_training'])
    print('final target train MSE:', metadata['final_target_train_mse'])
    print('MAE:', metrics['mae_percent'])
    print('RMSE:', metrics['rmse_percent'])
    print('MAX:', metrics['max_error_percent'])
    for figure in sorted((latest_diagnostic / 'figures').glob('*.png')):
        display(Image(filename=str(figure)))

## 9. Repository clean check

In [ ]:
status_short = subprocess.check_output(['git', 'status', '--short'], cwd=REPO_ROOT, text=True).strip()
print('Notebook execution should not modify tracked repository files.')
if status_short:
    print('Current working tree changes:')
    print(status_short)
else:
    print('Working tree clean.')

## 10. Formal single-pair gated execution

This launches the 515-epoch LG HG2 -> Samsung 30T formal run only when explicitly enabled.

In [ ]:

RUN_FORMAL_SINGLE_PAIR = False

FORMAL_CONFIG_EXPECTED = {
    'run_mode': 'formal',
    'epochs': 515,
    'batch_size': 512,
    'learning_rate': 0.01,
    'scheduler_step_size': 100,
    'scheduler_gamma': 0.5,
    'max_train_windows': None,
    'development_only': False,
    'verify_hashes': True,
    'resample_hz': 1.0,
}

def assert_formal_config(path: Path) -> dict:
    assert path.is_file(), 'Missing formal config: ' + str(path)
    config = yaml.safe_load(path.read_text())
    for key, expected in FORMAL_CONFIG_EXPECTED.items():
        actual = config.get(key)
        if actual != expected:
            raise AssertionError(
                'Formal config mismatch for '
                + key
                + ': expected '
                + repr(expected)
                + ', got '
                + repr(actual)
            )
    return config

if RUN_FORMAL_SINGLE_PAIR:
    current_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
    current_dirty = bool(subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_ROOT, text=True).strip())
    assert current_dirty is False, 'Formal run requires clean working tree.'
    assert torch.cuda.is_available() is True, 'Formal run should use CUDA.'
    assert SOURCE == 'lg_hg2'
    assert TARGET == 'samsung_30t'
    assert SOURCE_CHECKPOINT.is_file(), 'Missing source checkpoint.'
    source_payload = torch.load(SOURCE_CHECKPOINT, map_location='cpu', weights_only=False)
    assert int(source_payload.get('epoch')) == 514, 'Source checkpoint must be zero-based epoch 514.'
    formal_config_path = REPO_ROOT / 'configs/experiments/vidal_case2.yaml'
    formal_config = assert_formal_config(formal_config_path)
    print('formal config verified:', formal_config_path)
    target_baseline = formal[formal['dataset'] == TARGET]
    assert not target_baseline.empty, 'Missing target Case 1 formal baseline.'
    print('Launching formal run at commit:', current_commit)
    subprocess.run([
        sys.executable, 'scripts/run_vidal_case2.py',
        '--source', 'lg_hg2',
        '--target', 'samsung_30t',
        '--config', str(formal_config_path),
        '--input-root', str(VIDAL_V2_ROOT),
        '--output-root', str(CASE2_OUTPUT_ROOT),
        '--case1-output-root', str(CASE1_OUTPUT_ROOT),
        '--source-checkpoint', str(SOURCE_CHECKPOINT),
        '--cache-root', str(LOCAL_CACHE_ROOT),
        '--run-mode', formal_config['run_mode'],
        '--device', 'auto',
    ], cwd=REPO_ROOT, check=True, text=True)
else:
    print('Set RUN_FORMAL_SINGLE_PAIR = True to launch the 515-epoch LG HG2 -> Samsung 30T formal run.')


## 11. Formal single-pair result inspection

In [ ]:

formal_pair_root = CASE2_OUTPUT_ROOT / 'source_lg_hg2__target_samsung_30t' / 'seed_42'
formal_runs = []
for meta_path in sorted(formal_pair_root.glob('*/run_metadata.json')):
    run = meta_path.parent
    metrics_path = run / 'metrics.json'
    comparison_path = run / 'comparison_to_case1.json'
    metadata = json.loads(meta_path.read_text())
    source_meta = metadata.get('source_checkpoint', {})
    if (
        metadata.get('run_mode') == 'formal'
        and metadata.get('development_only') is False
        and metadata.get('source_dataset') == 'lg_hg2'
        and metadata.get('target_dataset') == 'samsung_30t'
        and source_meta.get('source_checkpoint_epoch') == 514
        and metadata.get('git_dirty') is False
        and metadata.get('scheduler_last_epoch') == 515
        and metrics_path.is_file()
        and comparison_path.is_file()
        and meta_path.is_file()
    ):
        formal_runs.append(run)

if not formal_runs:
    print('No completed formal LG HG2 -> Samsung 30T run found yet.')
else:
    latest_formal = formal_runs[-1]
    metadata = json.loads((latest_formal / 'run_metadata.json').read_text())
    metrics = json.loads((latest_formal / 'metrics.json').read_text())
    comparison = json.loads((latest_formal / 'comparison_to_case1.json').read_text())
    print('run path:', latest_formal)
    print('commit:', metadata['git_commit'])
    print('device:', metadata['device'])
    print('scheduler_last_epoch:', metadata.get('scheduler_last_epoch'))
    print('initial target MSE:', metadata['initial_target_mse_before_training'])
    print('final target train MSE:', metadata['final_target_train_mse'])
    print('Case 2 MAE/RMSE/MAX:', metrics['mae_percent'], metrics['rmse_percent'], metrics['max_error_percent'])
    print('Samsung Case 1 MAE/RMSE/MAX:', comparison.get('case1_target_mae_percent'), comparison.get('case1_target_rmse_percent'), comparison.get('case1_target_max_percent'))
    print('relative RMSE change:', comparison.get('rmse_relative_change_percent'))
    for figure in sorted((latest_formal / 'figures').glob('*.png')):
        display(Image(filename=str(figure)))


## 12. Formal all-pairs gated execution

This launches six directed 515-epoch formal runs only when explicitly enabled.

In [ ]:

RUN_FORMAL_ALL_PAIRS = False

if RUN_FORMAL_ALL_PAIRS:
    current_dirty = bool(subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_ROOT, text=True).strip())
    assert current_dirty is False, 'Formal all-pairs requires clean working tree.'
    assert torch.cuda.is_available() is True, 'Formal all-pairs should use CUDA.'
    required = {'panasonic_18650pf', 'lg_hg2', 'samsung_30t'}
    found = set(formal['dataset']) if not formal.empty else set()
    assert required <= found, 'Missing source checkpoints for: ' + str(sorted(required - found))
    subprocess.run([
        sys.executable, 'scripts/run_vidal_case2.py',
        '--all-pairs',
        '--config', 'configs/experiments/vidal_case2.yaml',
        '--input-root', str(VIDAL_V2_ROOT),
        '--output-root', str(CASE2_OUTPUT_ROOT),
        '--case1-output-root', str(CASE1_OUTPUT_ROOT),
        '--source-checkpoint-root', str(CASE1_OUTPUT_ROOT),
        '--cache-root', str(LOCAL_CACHE_ROOT),
        '--run-mode', 'formal',
        '--device', 'auto',
    ], cwd=REPO_ROOT, check=True, text=True)
else:
    print('Set RUN_FORMAL_ALL_PAIRS = True to launch all six 515-epoch Case 2 formal runs.')


## 13. Case 2 summary inspection

In [ ]:
summary_path = CASE2_OUTPUT_ROOT / 'case2_summary' / 'case2_summary.csv'
if summary_path.is_file():
    display(pd.read_csv(summary_path))
else:
    print('No formal Case 2 summary found yet:', summary_path)